## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

from sklearn.metrics import r2_score
from model_metrics import (
    summarize_model_performance,
    show_confusion_matrix,
    plot_3d_pdp,
)
from model_tuner import loadObjects
from eda_toolkit import ensure_directory

sys.path.append("../")

from core.functions import (
    create_shap_plots,
    plot_cumulative_fatalities_captured_journal,
)

import model_metrics

print(f"Model Metrics version: {model_metrics.__version__}")

In [ ]:
base_path = os.path.join(os.pardir)

# create image paths
image_path_png = os.path.join(base_path, "images", "png_images")
image_path_svg = os.path.join(base_path, "images", "svg_images")

# Use the function to ensure'data' directory exists
ensure_directory(image_path_png)
ensure_directory(image_path_svg)

## Read in the model objects

In [ ]:
model_linear_reg = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/619488654907862878/ea03902ec4ae4bb695f9aa2645e65159/artifacts/lr_log_fatalities/model.pkl"
)
model_lasso_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/619488654907862878/8956e1ff54734e57ab7700c6de3e25ed/artifacts/lasso_log_fatalities/model.pkl"
)
model_ridge_rfe = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/619488654907862878/9ffe3399ce254481a33f9bc32779c459/artifacts/ridge_log_fatalities/model.pkl"
)

model_xgb = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/619488654907862878/9dca6fb948964b24a6b1d2061886e7a6/artifacts/xgb_log_fatalities/model.pkl"
)
model_cat = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/models/619488654907862878/dda33f95c3a1492c9296cc42f4b70d8e/artifacts/cat_log_fatalities/model.pkl"
)

## Read in partioning data

In [ ]:
X_train = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_train.parquet"
)

In [ ]:
X_valid = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_valid.parquet"
)

In [ ]:
y_valid = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/y_valid_log_fatalities.parquet"
)

In [ ]:
X_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/X_test.parquet"
)

In [ ]:
y_test = pd.read_parquet(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/processed/y_test_log_fatalities.parquet"
)

## Read in original data for reference

In [ ]:
df = pd.read_csv(
    "/home/lshpaner/Python_Projects/acled_ukraine/data/raw/ACLED Data_2026-01-02.csv"
)

In [ ]:
y_pred = model_xgb.predict(X_test)

## Summarize model performance

In [ ]:
categorical_cols = [
    "admin1",
    "sub_event_type",
    "interaction",
    "source_scale",
    "geo_precision",
    "time_precision",
]

In [ ]:
print("Validation Set Results")
# Raw X_test for all models
X_valid_raw = X_valid
for col in categorical_cols:
    combined_cats = pd.Categorical(
        pd.concat([X_train[col], X_valid[col], X_valid[col]])
    ).categories
    X_valid[col] = pd.Categorical(X_valid[col], categories=combined_cats)

# Get predictions per model
valid_preds = {
    "Linear Regressor": model_linear_reg.predict(X_valid_raw),
    "Lasso RFE": model_lasso_rfe.predict(X_valid_raw),
    "XGBoost Regressor": model_xgb.predict(X_valid),
    "CatBoost Regressor": model_cat.predict(X_valid),
}

for name, pred in valid_preds.items():
    print(f"{name} R²: {r2_score(y_valid, pred):.4f}")

print()

print("Test Set Results")

# Raw X_test for all models
X_test_raw = X_test
for col in categorical_cols:
    combined_cats = pd.Categorical(
        pd.concat([X_train[col], X_valid[col], X_test[col]])
    ).categories
    X_test[col] = pd.Categorical(X_test[col], categories=combined_cats)

# Get predictions per model
test_preds = {
    "Linear Regressor": model_linear_reg.predict(X_test_raw),
    "Lasso RFE": model_lasso_rfe.predict(X_test_raw),
    "XGBoost Regressor": model_xgb.predict(X_test),
    "CatBoost Regressor": model_cat.predict(X_test),
}

for name, pred in test_preds.items():
    print(f"{name} R²: {r2_score(y_test, pred):.4f}")

In [ ]:
y_test["fatalities"] = np.expm1(y_test["log_fatalities"])

In [ ]:
y_test["fatalities"].sum()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from scipy.interpolate import make_interp_spline

fig, ax = plt.subplots(figsize=(7, 5))

# Curve
x_pts = np.array([0.0, 0.167, 0.333, 0.5, 0.667, 0.833, 1.0])
y_pts = np.array([0.0, 0.53, 0.70, 0.81, 0.89, 0.96, 1.0])
spl = make_interp_spline(x_pts, y_pts, k=3)
x_fine = np.linspace(0, 1, 400)
y_fine = spl(x_fine)

# ── Shaded AUC region ──────────────────────────────────────────────────────
ax.fill_between(x_fine, 0, y_fine, color="#1D9E75", alpha=0.10, zorder=1)

# ── One highlighted trapezoid (strip k) ───────────────────────────────────
k = 2
x0, x1 = x_pts[k], x_pts[k + 1]
y0, y1 = float(spl(x0)), float(spl(x1))

trap = Polygon(
    [(x0, 0), (x1, 0), (x1, y1), (x0, y0)],
    closed=True,
    facecolor="#AEC9E8",
    alpha=0.55,
    edgecolor="#2A6096",
    linewidth=1.2,
    zorder=2,
)
ax.add_patch(trap)

# Dashed reference lines
ax.plot([0, x0], [y0, y0], color="#2A6096", lw=0.9, ls="--", zorder=3)
ax.plot([0, x1], [y1, y1], color="#2A6096", lw=0.9, ls="--", zorder=3)
ax.plot([x0, x0], [0, y0], color="#2A6096", lw=0.9, ls=":", zorder=3)
ax.plot([x1, x1], [0, y1], color="#2A6096", lw=0.9, ls=":", zorder=3)

# Y-axis labels
ax.text(
    -0.03, y0, r"$F(x_{k-1})$", ha="right", va="center", color="#2A6096", fontsize=9
)
ax.text(-0.03, y1, r"$F(x_k)$", ha="right", va="center", color="#2A6096", fontsize=9)

# X-axis labels
ax.text(x0, -0.05, r"$x_{k-1}$", ha="center", va="top", color="#2A6096", fontsize=9)
ax.text(x1, -0.05, r"$x_k$", ha="center", va="top", color="#2A6096", fontsize=9)

# Area_k label
ax.text(
    (x0 + x1) / 2,
    (y0 + y1) / 2 - 0.05,
    r"Area$_k$",
    ha="center",
    va="center",
    color="#1A4A6B",
    fontsize=9,
    style="italic",
)

# ── Capture curve ──────────────────────────────────────────────────────────
ax.plot(x_fine, y_fine, color="#1D9E75", lw=2.2, label="Capture curve $F(x)$", zorder=4)

# ── Random baseline ────────────────────────────────────────────────────────
ax.plot(
    [0, 1],
    [0, 1],
    color="#888780",
    lw=1.3,
    ls=(0, (7, 4)),
    label="Random baseline (AUC = 0.5)",
    zorder=3,
)

# ── AUC annotation ─────────────────────────────────────────────────────────
ax.text(0.15, 0.35, "AUC", color="#1D9E75", fontsize=11, fontweight="bold", ha="center")
ax.text(0.15, 0.28, r"$=\int_0^1 F(x)\,dx$", color="#1D9E75", fontsize=9, ha="center")

# ── Axes ───────────────────────────────────────────────────────────────────
ax.set_xlim(-0.06, 1.05)
ax.set_ylim(-0.10, 1.06)
ax.set_xticks([0, 0.25, 0.50, 0.75, 1.00])
ax.set_yticks([0, 0.25, 0.50, 0.75, 1.00])
ax.set_xlabel("Fraction of events inspected, $x$", fontsize=11)
ax.set_ylabel("Fraction of fatalities captured, $F(x)$", fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(labelsize=9)
ax.grid(True, alpha=0.2, linestyle="--", linewidth=0.5)
ax.legend(loc="lower right", fontsize=9, framealpha=0.9, edgecolor="#cccccc")

plt.tight_layout()
plt.savefig(
    os.path.join(image_path_png, "capture_curve_trapezoidal.png"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.savefig(
    os.path.join(image_path_svg, "capture_curve_trapezoidal.svg"),
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

In [ ]:
plot_cumulative_fatalities_captured_journal(
    y_test["log_fatalities"],
    model_xgb.predict(X_test),
    title="",
    dpi=100,
    model_name="XGBoost",
    save_path="../images/svg_images/cumulative_fatalities_captured.svg",
)

In [ ]:
regression_metrics = summarize_model_performance(
    model=[model_linear_reg, model_lasso_rfe, model_xgb, model_cat],
    model_title=[
        "Linear Regressor",
        "Lasso RFE",
        "XGBoost Regressor",
        "CatBoost Regressor",
    ],
    X=X_valid_raw,
    y=y_valid,
    y_pred=[
        model_linear_reg.predict(X_valid_raw),
        model_lasso_rfe.predict(X_valid_raw),
        model_xgb.predict(X_valid),
        model_cat.predict(X_valid),
    ],
    model_type="regression",
    return_df=True,
    decimal_places=3,
    include_adjusted_r2=True,
)
print("Validation Set Performance Metrics:")
regression_metrics

In [ ]:
y_test["log_fatalities"]

In [ ]:
X_test.shape

In [ ]:
from model_tuner import report_model_metrics

report_model_metrics(model_xgb, X_test, y_test["log_fatalities"])

In [ ]:
regression_metrics = summarize_model_performance(
    model=[model_linear_reg, model_lasso_rfe, model_xgb, model_cat],
    model_title=[
        "Linear Regressor",
        "Lasso RFE",
        "XGBoost Regressor",
        "CatBoost Regressor",
    ],
    X=X_test_raw,
    y=y_test,
    y_pred=[
        model_linear_reg.predict(X_test_raw),
        model_lasso_rfe.predict(X_test_raw),
        model_xgb.predict(X_test),
        model_cat.predict(X_test),
    ],
    model_type="regression",
    return_df=True,
    decimal_places=3,
    include_adjusted_r2=True,
)

print("Test Set Performance Metrics:")

regression_metrics

In [ ]:
X_test.shape

In [ ]:
from model_tuner import evaluate_bootstrap_metrics

In [ ]:
from sklearn.metrics import r2_score

metrics = [
    "r2",
    "adjusted_r2",
    "neg_root_mean_squared_error",
    "neg_mean_squared_error",
    "neg_mean_absolute_error",
    "explained_variance",
]

test_inputs = {
    "Linear Regressor": (model_linear_reg, X_test_raw),
    "Lasso RFE": (model_lasso_rfe, X_test_raw),
    "XGBoost Regressor": (model_xgb, X_test),
    "CatBoost Regressor": (model_cat, X_test),
}

bootstrap_results = {}

for name, (model, X) in test_inputs.items():
    print(f"\nRunning bootstrap for: {name}")
    results = evaluate_bootstrap_metrics(
        model=model,
        X=X,
        y=y_test,
        y_pred_prob=None,
        n_samples=5000,
        num_resamples=5000,
        metrics=metrics,
        random_state=42,
        average="macro",
        thresholds=None,
        model_type="regression",
        stratify=None,
        balance=False,
        class_proportions=None,
    )
    results.insert(0, "Model", name)
    bootstrap_results[name] = results

# Display each individually
for name, df in bootstrap_results.items():
    print(f"\n{name}")
    display(round(df, 3))

# Or combine into one table
all_results = pd.concat(bootstrap_results.values(), ignore_index=True)

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": model_linear_reg.get_feature_names(),
        "coef": model_linear_reg.estimator.named_steps["lr"].coef_,
    }
).sort_values("coef", key=abs, ascending=False)

coef_df.head(20)

In [ ]:
X_columns_list = loadObjects(
    "/home/lshpaner/Python_Projects/acled_ukraine/mlruns/preprocessing/459884901257813393/6812867e15454f0a987c4e2ffa3e1c7f/artifacts/X_columns_list.pkl"
)

In [ ]:
numerical_cols = [col for col in X_columns_list if col not in categorical_cols]

In [ ]:
xgb_estimator = model_xgb.estimator[-1]

## Partial Dependence Plot (PDP)

In [ ]:
label_map = (None,)
apply_label_map = (True,)


label_replacements = {
    # Sub Event Types
    "Shelling/artillery/missile attack": "Shelling / Missile",
    "Peaceful protest": "Peaceful protest",
    "Mob violence": "Mob violence",
    "Government regains territory": "Gov regains territory",
    "Change to group/activity": "Group change",
    "Armed clash": "Armed clash",
    "Abduction/forced disappearance": "Forced disappearance",
    # Source Scale
    "Subnational-Regional": "Subnational Regional",
    "Subnational": "Subnational",
    "Other-Subnational": "Other Subnational",
    "Other-National": "Other National",
    "National": "National",
    "Other-International": "Other International",
    "International": "International",
    "New media-National": "New media National",
    "New media-International": "New media International",
}

In [ ]:
source_scale_map = {
    "International": "Intl",
    "Local partner-New media": "Local+Media",
    "National": "National",
    "National-International": "Nat+Intl",
    "National-Regional": "Nat+Reg",
    "New media": "Media",
    "New media-National": "Media+Nat",
    "Other": "Other",
    "Other-International": "Other+Intl",
    "Other-National": "Other+Nat",
    "Other-New media": "Other+Media",
    "Other-Regional": "Other+Reg",
    "Other-Subnational": "Other+Subnat",
    "Regional": "Regional",
    "Subnational": "Subnat",
    "Subnational-National": "Subnat+Nat",
    "Subnational-Regional": "Subnat+Reg",
}

event_map = {
    "Abduction/forced disappearance": "Abduction",
    "Agreement": "Agreement",
    "Air/drone strike": "Airstrike",
    "Armed clash": "Armed Clash",
    "Arrests": "Arrests",
    "Attack": "Attack",
    "Change to group/activity": "Group Change",
    "Chemical weapon": "Chemical",
    "Disrupted weapons use": "Disrupted Use",
    "Government regains territory": "Govt Gains",
    "Grenade": "Grenade",
    "Looting/property destruction": "Looting",
    "Mob violence": "Mob Violence",
    "Non-state actor overtakes territory": "NSA Gains",
    "Other": "Other",
    "Peaceful protest": "Peaceful Protest",
    "Remote explosive/landmine/IED": "IED",
    "Sexual violence": "Sexual Violence",
    "Shelling/artillery/missile attack": "Shelling",
    "Suicide bomb": "Suicide Bomb",
    "Violent demonstration": "Violent Protest",
}

In [ ]:
plot_3d_pdp(
    model=model_xgb.estimator,
    dataframe=X_test,
    feature_names=["sub_event_type", "source_scale"],
    x_label="",
    y_label="",
    x_label_map=event_map,
    y_label_map=source_scale_map,
    matplotlib_colormap="viridis",
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="static",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=35,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    figsize=(15, 8),
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    view_angle=(23, 50),
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=False,
    enable_zoom=True,
    save_plots="html",
)

In [ ]:
model_xgb.get_feature_names()

In [ ]:
# Get bare estimator and transformed data
xgb_estimator = model_xgb.estimator[-1]
preprocessor = model_xgb.estimator[:-1]

feature_names_out = preprocessor.get_feature_names_out()
X_test_transformed = pd.DataFrame(
    (
        preprocessor.transform(X_test).toarray()
        if hasattr(preprocessor.transform(X_test), "toarray")
        else preprocessor.transform(X_test)
    ),
    columns=feature_names_out,
    index=X_test.index,
)

# Pick one representative OHE column per category group
# e.g. use the index of the category in the OHE expansion
sub_event_cols = [f for f in feature_names_out if "sub_event_type" in f]
source_scale_cols = [f for f in feature_names_out if "source_scale" in f]

print("sub_event_type OHE cols:", sub_event_cols)
print("source_scale OHE cols:", source_scale_cols)

In [ ]:
from model_metrics import plot_3d_pdp

plot_3d_pdp(
    model=xgb_estimator,
    dataframe=X_test_transformed,
    feature_names=[sub_event_cols[0], source_scale_cols[0]],
    x_label="",
    y_label="",
    # x_label="sub_event_type",
    # y_label="source_scale",
    x_label_map=event_map,
    y_label_map=source_scale_map,
    z_label="Partial Dependence",
    title="3D Partial Dependence Plot of Event Type vs. Source Scale",
    html_file_path="../data",
    # image_filename="3d_pdp",
    html_file_name="3d_pdp.html",
    image_path_svg="../data",
    plot_type="interactive",
    text_wrap=80,
    zoom_out_factor=1.3,
    grid_resolution=25,
    label_fontsize=8,
    tick_fontsize=6,
    title_x=0.38,
    top_margin=20,
    # bottom_margin=120,
    right_margin=80,
    left_margin=70,
    cbar_x=0.9,
    cbar_thickness=25,
    show_modebar=True,
    enable_zoom=True,
    save_plots="html",
    modebar_image_format="svg",
)

## Save Out Predictions from Best Model

In [ ]:
y_test.rename(columns={"log_fatalities": "actual_log_fatalities"}, inplace=True)

In [ ]:
X_test.head()

In [ ]:
predictions_x_test = pd.Series(
    model_xgb.predict(X_test), index=X_test.index, name="prediction"
)

In [ ]:
r2_score(
    np.expm1(y_test).round(0).astype(int),
    np.expm1(predictions_x_test).round(0).astype(int),
)

In [ ]:
r2_score(y_test.to_numpy(), predictions_x_test.to_numpy())

In [ ]:
log_preds = predictions_x_test.to_frame(name="log_pred_fatalities")
log_preds["pred_fatalities"] = (
    np.expm1(log_preds["log_pred_fatalities"]).round(0).astype(int)
)

# Use X_test_raw for metadata join
log_preds_merge = log_preds.join(X_test_raw, how="inner")

# Back-transform y_test
log_preds_merge["actual_fatalities"] = np.expm1(y_test).round(0).astype(int)

log_preds_merge[["actual_fatalities", "pred_fatalities"]]

In [ ]:
from sklearn.metrics import r2_score

r2_score(log_preds_merge["actual_fatalities"], log_preds_merge["pred_fatalities"])

In [ ]:
print(log_preds_merge["actual_fatalities"].describe())
print(
    f"\nEvents with >10 actual fatalities: {(log_preds_merge['actual_fatalities'] > 10).sum()}"
)
print(
    f"Events with >50 actual fatalities: {(log_preds_merge['actual_fatalities'] > 50).sum()}"
)

## Non-Zero Fatalities

In [ ]:
mask = (log_preds_merge["actual_fatalities"] > 0) | (
    log_preds_merge["pred_fatalities"] > 0
)
nonzero = log_preds_merge[mask]

print(f"Non-zero subset: {mask.sum()} of {len(mask)} events ({mask.mean()*100:.1f}%)")
print(
    f"Non-zero R²: {r2_score(nonzero['actual_fatalities'], nonzero['pred_fatalities']):.4f}"
)

In [ ]:
from eda_toolkit import scatter_fit_plot

scatter_fit_plot(
    df=log_preds_merge,
    x_vars=["actual_fatalities"],
    y_vars=["pred_fatalities"],
    show_legend=True,
    show_plot="subplots",
    subplot_figsize=None,
    label_fontsize=14,
    tick_fontsize=12,
    add_best_fit_line=True,
    scatter_color="#808080",
    show_correlation=True,
)

In [ ]:
log_preds_merge.head()

### Save predictions to path

In [ ]:
log_preds_merge.to_csv(
    os.path.join("..", "data", "processed", "log_preds_merge.csv"), index=False
)

## Failure Mode Analysis

In [ ]:
y_test_ground_truth_actual = np.where(y_test > 0, 1, 0).ravel()

In [ ]:
pred_fatalities = log_preds_merge["pred_fatalities"].to_numpy()
pred_fatalities

In [ ]:
show_confusion_matrix(
    y=y_test_ground_truth_actual,
    y_prob=pred_fatalities,
    model_title="XGBoost",
    cmap="Blues",
    text_wrap=50,
    subplots=True,
    n_rows=1,
    figsize=(6, 6),
)